PHASE 1:


1.   Hugging Face Transformers
2.   Stanza (EN+HI)
3.   WordNet
4.   Evaluation Tools
5.   DBpedia query support



In [ ]:
# Core NLP & ML libraries
!pip install -q transformers datasets sentencepiece sacrebleu nltk stanza spacy

# For evaluation and utilities
!pip install -q evaluate rouge-score

# Optional: DBpedia querying
!pip install -q SPARQLWrapper


In [ ]:
import stanza

# Download models (run once)
stanza.download('en')
stanza.download('hi')


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/stanza_resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Downloading default packages for language: hi (Hindi) ...
INFO:stanza:File exists: /root/stanza_resources/hi/default.zip
INFO:stanza:Finished downloading models and saved to /root/stanza_resources


In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

print("Transformers setup complete.")


Transformers setup complete.


PHASE 2:


1.   Loaded Samantar (EN-HI)
2.   Cleaned and filtered sentence pairs
3.   Created a reusable dataset structure



In [ ]:
from datasets import load_dataset
import re
import pandas as pd


In [ ]:
!unzip -q train.en.zip
!unzip -q train.hi.zip

replace train.en? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y
replace train.hi? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
y
y


In [ ]:
with open("train.en", "r", encoding="utf-8") as f:
    en_lines = f.readlines()

with open("train.hi", "r", encoding="utf-8") as f:
    hi_lines = f.readlines()

print("English lines:", len(en_lines))
print("Hindi lines:", len(hi_lines))


English lines: 8568307
Hindi lines: 8568307


In [ ]:
import re

def clean_text(text):
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


In [ ]:
MAX_SAMPLES = 10000  # safe for Colab

parallel_data = []

for en, hi in zip(en_lines[:MAX_SAMPLES], hi_lines[:MAX_SAMPLES]):
    en_clean = clean_text(en)
    hi_clean = clean_text(hi)

    if len(en_clean.split()) > 2 and len(hi_clean.split()) > 2:
        parallel_data.append({
            "en": en_clean,
            "hi": hi_clean
        })

print("Total usable sentence pairs:", len(parallel_data))


Total usable sentence pairs: 9774


In [ ]:
import pandas as pd

df = pd.DataFrame(parallel_data)
df.head()


,en,hi
0,"In reply, Pakistan got off to a solid start.",जिसके जवाब में पाक ने अच्छी शुरुआत की थी.
1,The European Union has seven principal decisio...,यूरोपीय संघ के महत्वपूर्ण संस्थानों में यूरोपि...
2,The Congress leader represents Sivaganga Lok S...,कांग्रेस नेता तमिलनाडु से शिवगंगा लोकसभा क्षेत...
3,Prompt the user about connection attempts,संबंधन प्रयास के बारे में उपयोक्ता को प्रांप्ट...
4,"Further, the Minister announced that Deposit I...",वित्त मंत्री ने घोषणा कि जमा बीमा और ऋण गारंटी...


In [ ]:
idx = 16
print("EN:", df.iloc[idx]["en"])
print("HI:", df.iloc[idx]["hi"])


EN: Then after it becomes dry, rinse off with water.
HI: इसके बाद जब य सुख जाए तो इसे साफ पानी से धो लें।


PHASE 3:
For each English-Hindi sentence pair, we will extract:


*   Tokens
*   POS tags
*   Named Entities
*   Dependency relations
and store in a structured format for later rule integration.



1.   English & HIndi linguistic annotation
2.   POS tags, NER, dependencies extracted
3.   Structured format for rule layer
4.   Ready for semantic & entity knowledge integration







In [ ]:
import stanza

# English pipeline
nlp_en = stanza.Pipeline(
    lang='en',
    processors='tokenize,pos,lemma,ner,depparse',
    use_gpu=True
)

# Hindi pipeline
nlp_hi = stanza.Pipeline(
    lang='hi',
    processors='tokenize,pos,lemma,ner,depparse',
    use_gpu=True
)


INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package                   |
-----------------------------------------
| tokenize  | combined                  |
| mwt       | combined                  |
| pos       | combined_charlm           |
| lemma     | combined_nocharlm         |
| depparse  | combined_charlm           |
| ner       | ontonotes-ww-multi_charlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: hi (Hindi):
| Processor | Package       |
-----------------------------
| tokenize  | hdtb          |
| pos       | hdtb_charlm   |
| lemma     | hdtb_nocharlm |
| depparse  | hdtb_charlm   |
| ner       | ilner_charlm  |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [ ]:
def annotate_sentence(sentence, nlp_pipeline):
    doc = nlp_pipeline(sentence)

    tokens = []
    entities = []
    dependencies = []

    for sent in doc.sentences:
        # Tokens + POS
        for word in sent.words:
            tokens.append({
                "text": word.text,
                "lemma": word.lemma,
                "upos": word.upos,
                "xpos": word.xpos,
                "feats": word.feats if word.feats else {},
                "head": word.head,
                "deprel": word.deprel
            })

        # Named Entities
        for ent in sent.ents:
            entities.append({
                "text": ent.text,
                "type": ent.type
            })

        # Dependency edges
        for word in sent.words:
            if word.head != 0:
                dependencies.append({
                    "governor": sent.words[word.head - 1].text,
                    "dependent": word.text,
                    "relation": word.deprel
                })

    return {
        "tokens": tokens,
        "entities": entities,
        "dependencies": dependencies
    }


In [ ]:
sample_idx = 5

en_sentence = df.iloc[sample_idx]["en"]
hi_sentence = df.iloc[sample_idx]["hi"]

en_annotation = annotate_sentence(en_sentence, nlp_en)
hi_annotation = annotate_sentence(hi_sentence, nlp_hi)

print("ENGLISH SENTENCE:")
print(en_sentence)
print("\nENGLISH ENTITIES:")
print(en_annotation["entities"])

print("\nHINDI SENTENCE:")
print(hi_sentence)
print("\nHINDI ENTITIES:")
print(hi_annotation["entities"])


ENGLISH SENTENCE:
Therefore, brothers, be more diligent to make your calling and election sure. For if you do these things, you will never stumble.

ENGLISH ENTITIES:
[]

HINDI SENTENCE:
इस कारण हे भाइयों, अपने बुलाए जाने, और चुन लिये जाने को सिद्ध करने का भली भांति यत्न करते जाओ, क्योंकि यदि ऐसा करोगे, तो कभी भी ठोकर न खाओगे।

HINDI ENTITIES:
[]


In [ ]:
from tqdm import tqdm

ANNOTATION_LIMIT = 2000  # safe for Colab -> 2000

annotated_data = []

for idx in tqdm(range(ANNOTATION_LIMIT)):
    en_sent = df.iloc[idx]["en"]
    hi_sent = df.iloc[idx]["hi"]

    en_ann = annotate_sentence(en_sent, nlp_en)
    hi_ann = annotate_sentence(hi_sent, nlp_hi)

    annotated_data.append({
        "en": en_sent,
        "hi": hi_sent,
        "en_annotation": en_ann,
        "hi_annotation": hi_ann
    })


100%|██████████| 2000/2000 [1:17:23<00:00,  2.32s/it]


In [ ]:
annotated_data[0].keys()


dict_keys(['en', 'hi', 'en_annotation', 'hi_annotation'])

In [ ]:
annotated_data[0]["en_annotation"]["dependencies"][:5]


[{'governor': 'reply', 'dependent': 'In', 'relation': 'case'},
 {'governor': 'got', 'dependent': 'reply', 'relation': 'obl'},
 {'governor': 'reply', 'dependent': ',', 'relation': 'punct'},
 {'governor': 'got', 'dependent': 'Pakistan', 'relation': 'nsubj'},
 {'governor': 'got', 'dependent': 'off', 'relation': 'compound:prt'}]

In [ ]:
annotated_data[0]["en_annotation"]["tokens"][0]


{'text': 'In',
 'lemma': 'in',
 'upos': 'ADP',
 'xpos': 'IN',
 'feats': {},
 'head': 2,
 'deprel': 'case'}

PHASE 4: Semantic & Knowledge Integration

PHASE 4.1: WordNet-Semantic Disambiguation


*   NMT often mistranslates polysemous words
*   WordNet helps choose the correct sense





In [ ]:
from nltk.corpus import wordnet as wn


In [ ]:
def get_wordnet_senses(word, pos_tag=None):
    """
    Returns WordNet senses for a word.
    """
    if pos_tag:
        return wn.synsets(word, pos=pos_tag)
    return wn.synsets(word)


In [ ]:
def map_pos_to_wordnet(upos):
    if upos == 'NOUN':
        return wn.NOUN
    if upos == 'VERB':
        return wn.VERB
    if upos == 'ADJ':
        return wn.ADJ
    if upos == 'ADV':
        return wn.ADV
    return None


In [ ]:
def enrich_with_wordnet(tokens):
    enriched_tokens = []

    for tok in tokens:
        wn_pos = map_pos_to_wordnet(tok["upos"])
        senses = []

        if wn_pos:
            synsets = get_wordnet_senses(tok["lemma"], wn_pos)
            senses = [syn.name() for syn in synsets[:3]]  # top 3 senses

        tok["wordnet_senses"] = senses
        enriched_tokens.append(tok)

    return enriched_tokens


In [ ]:
sample_en_tokens = annotated_data[0]["en_annotation"]["tokens"]
enriched_tokens = enrich_with_wordnet(sample_en_tokens)

for t in enriched_tokens[:5]:
    print(t["text"], "→", t["wordnet_senses"])


In → []
reply → ['answer.n.01', 'reply.n.02']
, → []
Pakistan → []
got → ['get.v.01', 'become.v.01', 'get.v.03']


PHASE 4.2: DBpedia-Entity Linking


*   Preserve names, places, organizations
*   Avoid mistranslation of proper nouns
*   Enable annotated output





In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper("https://dbpedia.org/sparql")


In [ ]:
def dbpedia_lookup(entity_text):
    query = f"""
    SELECT ?entity WHERE {{
      ?entity rdfs:label "{entity_text}"@en .
    }} LIMIT 1
    """

    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        results = sparql.query().convert()
        bindings = results["results"]["bindings"]
        if bindings:
            return bindings[0]["entity"]["value"]
    except:
        pass

    return None


In [ ]:
def link_entities_dbpedia(entities):
    linked_entities = []

    for ent in entities:
        uri = dbpedia_lookup(ent["text"])
        ent["dbpedia_uri"] = uri
        linked_entities.append(ent)

    return linked_entities


In [ ]:
sample_entities = annotated_data[0]["en_annotation"]["entities"]
linked_entities = link_entities_dbpedia(sample_entities)

for e in linked_entities:
    print(e)


{'text': 'Pakistan', 'type': 'GPE', 'dbpedia_uri': 'http://dbpedia.org/resource/Category:Pakistan'}


PHASE 4.3: Integrate Knowledge into Annotations

In [ ]:
def enrich_annotation(annotation):
    annotation["tokens"] = enrich_with_wordnet(annotation["tokens"])
    annotation["entities"] = link_entities_dbpedia(annotation["entities"])
    return annotation


In [ ]:
KNOWLEDGE_LIMIT = 500 #500

for i in range(KNOWLEDGE_LIMIT):
    annotated_data[i]["en_annotation"] = enrich_annotation(
        annotated_data[i]["en_annotation"]
    )


PHASE 5: Rule-Augmented Translation Layer

Injecting linguistic rules between:
Semantic analysis -> Neural Translation

English Sentence -> Stanza + WordNet + DBpedia -> Rule Engine -> Constrained/Guided NMT -> Annotated Hindi Output



1.   Syntax-aware reordering
2.   Knowledge-aware entity preservation
3.   Morphological Guidance
4.   Implementing Rule Engine



In [ ]:
def extract_svo(tokens):
    subject = []
    obj = []
    verb = []

    for tok in tokens:
        if tok["deprel"] in ["nsubj", "nsubj:pass"]:
            subject.append(tok)
        elif tok["deprel"] in ["obj", "iobj"]:
            obj.append(tok)
        elif tok["upos"] == "VERB":
            verb.append(tok)

    return subject, obj, verb


In [ ]:
def apply_sov_reordering(tokens):
    subj, obj, verb = extract_svo(tokens)

    reordered = subj + obj + verb

    # Add remaining tokens (adverbs, auxiliaries, etc.)
    remaining = [
        t for t in tokens
        if t not in reordered
    ]

    return reordered + remaining


In [ ]:
sample_tokens = annotated_data[0]["en_annotation"]["tokens"]
reordered_tokens = apply_sov_reordering(sample_tokens)

print("Original:", " ".join(t["text"] for t in sample_tokens))
print("Reordered:", " ".join(t["text"] for t in reordered_tokens))


Original: In reply , Pakistan got off to a solid start .
Reordered: Pakistan got In reply , off to a solid start .


In [ ]:
def mask_entities(tokens, entities):
    entity_map = {}
    masked_tokens = []

    for i, ent in enumerate(entities):
        placeholder = f"<ENTITY_{i}>"
        entity_map[placeholder] = ent["text"]

    for tok in tokens:
        replaced = False
        for placeholder, text in entity_map.items():
            if tok["text"] == text:
                masked_tokens.append(placeholder)
                replaced = True
                break
        if not replaced:
            masked_tokens.append(tok["text"])

    return masked_tokens, entity_map


In [ ]:
def unmask_entities(translated_text, entity_map):
    for placeholder, value in entity_map.items():
        translated_text = translated_text.replace(placeholder, value)
    return translated_text


In [ ]:
def parse_ud_feats(feat_string):
    """
    Convert UD feature string into dictionary.
    Example:
    'Tense=Past|Number=Sing' → {'Tense': 'Past', 'Number': 'Sing'}
    """
    feats = {}

    if feat_string is None or feat_string == "":
        return feats

    if isinstance(feat_string, str):
        for item in feat_string.split("|"):
            if "=" in item:
                key, value = item.split("=")
                feats[key] = value

    return feats


In [ ]:
def extract_agreement_hints(tokens):
    hints = []

    for tok in tokens:
        if tok["upos"] == "VERB":
            feat_dict = parse_ud_feats(tok.get("feats"))

            hints.append({
                "verb": tok["lemma"],
                "tense": feat_dict.get("Tense"),
                "number": feat_dict.get("Number"),
                "person": feat_dict.get("Person")
            })

    return hints


In [ ]:
def apply_rule_engine(annotation):
    tokens = annotation["tokens"]
    entities = annotation["entities"]

    # Step 1: Reordering
    reordered_tokens = apply_sov_reordering(tokens)

    # Step 2: Entity masking
    masked_sentence, entity_map = mask_entities(
        reordered_tokens, entities
    )

    # Step 3: Agreement hints
    agreement_hints = extract_agreement_hints(tokens)

    return {
        "rule_based_sentence": " ".join(masked_sentence),
        "entity_map": entity_map,
        "agreement_hints": agreement_hints
    }


In [ ]:
rule_output = apply_rule_engine(
    annotated_data[0]["en_annotation"]
)

print(rule_output)


{'rule_based_sentence': '<ENTITY_0> got In reply , off to a solid start .', 'entity_map': {'<ENTITY_0>': 'Pakistan'}, 'agreement_hints': [{'verb': 'get', 'tense': 'Past', 'number': 'Sing', 'person': '3'}]}


PHASE 6: Rule-Augmented Neural Machine TRanslation

English Sentence -> Linguistic + Semantic Analysis -> Rule Engine (SOV + Entity Masking) -> TRansformer (HF Marian/Indic model) -> Unmask Entities -> Final Hindi Translation (+ annotations)

Evaluation Metrics:


1.   BLEU
2.   METEOR
3.   chrF



In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch

model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(61950, 512, padding_idx=61949)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(61950, 512, padding_idx=61949)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

In [ ]:
def translate_with_rules(rule_output):
    sentence = rule_output["rule_based_sentence"]
    entity_map = rule_output["entity_map"]

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        translated = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(
        translated[0],
        skip_special_tokens=True
    )

    # Unmask entities
    final_translation = unmask_entities(decoded, entity_map)

    return final_translation


In [ ]:
sample = annotated_data[0]

rule_output = apply_rule_engine(sample["en_annotation"])

final_hi = translate_with_rules(rule_output)

print("ENGLISH:", sample["en"])
print("RULE INPUT:", rule_output["rule_based_sentence"])
print("HINDI OUTPUT:", final_hi)
print("GOLD HINDI:", sample["hi"])


ENGLISH: In reply, Pakistan got off to a solid start.
RULE INPUT: <ENTITY_0> got In reply , off to a solid start .
HINDI OUTPUT: <AR_0> जवाब में मिला, ठोस शुरुआत के लिए.
GOLD HINDI: जिसके जवाब में पाक ने अच्छी शुरुआत की थी.


In [ ]:
predictions = []
references = []

EVAL_LIMIT = 500 #500

for i in range(EVAL_LIMIT):
    rule_output = apply_rule_engine(
        annotated_data[i]["en_annotation"]
    )
    pred = translate_with_rules(rule_output)

    predictions.append(pred)
    references.append([annotated_data[i]["hi"]])  # sacreBLEU format


In [ ]:
!pip install sacrebleu nltk


In [ ]:
import sacrebleu


In [ ]:
bleu = sacrebleu.corpus_bleu(predictions, references)
chrf = sacrebleu.corpus_chrf(predictions, references)

print("BLEU:", bleu.score)
print("chrF:", chrf.score)


BLEU: 10.571070857151538
chrF: 32.01680970278022


In [ ]:
from nltk.translate.meteor_score import meteor_score


In [ ]:
meteor_scores = []

for pred, ref in zip(predictions, references):
    # ref is a list → take first reference
    ref_tokens = ref[0].split()
    pred_tokens = pred.split()

    score = meteor_score([ref_tokens], pred_tokens)
    meteor_scores.append(score)

print("METEOR:", sum(meteor_scores) / len(meteor_scores))


METEOR: 0.19091046752106847


In [ ]:
final_outputs = []
# 10
for i in range(10):
    rule_output = apply_rule_engine(
        annotated_data[i]["en_annotation"]
    )

    final_outputs.append({
        "english": annotated_data[i]["en"],
        "rule_input": rule_output["rule_based_sentence"],
        "agreement_hints": rule_output["agreement_hints"],
        "predicted_hindi": translate_with_rules(rule_output),
        "gold_hindi": annotated_data[i]["hi"]
    })

final_outputs[0]


{'english': 'In reply, Pakistan got off to a solid start.',
 'rule_input': '<ENTITY_0> got In reply , off to a solid start .',
 'agreement_hints': [{'verb': 'get',
   'tense': 'Past',
   'number': 'Sing',
   'person': '3'}],
 'predicted_hindi': '<AR_0> जवाब में मिला, ठोस शुरुआत के लिए.',
 'gold_hindi': 'जिसके जवाब में पाक ने अच्छी शुरुआत की थी.'}